# 125 — Router y especialistas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Router**: `route: entrada → {e₁,…,eₙ, fallback}` + `dispatch`. Clasifica y despacha
cada entrada al **especialista** adecuado (prompt, herramientas y modelo propios).
La ventaja es separación de preocupaciones: cada prompt se optimiza sin degradar a los
demás.

**Cota de rendimiento**: éxito global ≤ `p_router × p_especialista`. El router es un
clasificador: se evalúa con matriz de confusión y cobertura por clase, no con
impresiones.

**Diseños**: reglas (barato, auditable, frágil), LLM de una pasada (flexible, confianza
sin calibrar), cascada reglas→LLM (mejor coste/exactitud media). Siempre con umbral de
confianza y **fallback** explícito.


## 🧮 Ejemplo de referencia

Router de mesa de ayuda sobre 200 tickets:

```text
aciertos: facturación 72/80, técnico 88/92, ventas 20/28
exactitud = 180/200 = 0.90
techo del sistema = 0.90 × 0.92 (especialista) ≈ 0.828
```

La clase minoritaria (`ventas`, cobertura 0.714) es la que arrastra el sistema: el
diagnóstico sale de la matriz de confusión, no del promedio global.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("multiagent", seed=125)
show(result)


## Reflexión

1. El laboratorio consolida tres workers fijos; un router elegiría *uno*. ¿Qué información de la entrada necesitarías para decidir entre `quality`, `security` y `documentation`, y qué harías con confianza < umbral?
2. Si la exactitud del router es 0.90 y la del especialista 0.92, el techo es ≈0.828. ¿En cuál de los dos invertirías el siguiente esfuerzo de mejora y con qué evidencia lo decidirías?
3. ¿Por qué "añadir un especialista nuevo" puede *bajar* la exactitud global aunque ese especialista sea excelente?
